# M5.1 · OpenVINO CPU 文本编码器 —— 恢复与公平 A/B

**学习目标：** 验证 OpenVINO BF16 CPU 编码器的恢复（A 部分：本地编码器速度）、真实的端到端架构收益（B 部分：CPU TE 卸载的 GPU 流水线），以及区分架构效应与客户端监控效应的**公平因子实验**（C 部分）。编码器加速比 ≠ 图像吞吐收益——永远不要混淆。

参考讲义：[M5.1 讲义](../course/M05_parallel_heterogeneous/M5_1_OPENVINO_CPU_TE.zh-CN.md) · [English](./06_openvino_and_fair_ab.ipynb)

In [ ]:
from pathlib import Path
import json
import statistics
import sys
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "lab/reference_runs.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open Jupyter from the course repository root (see notebooks/README.md).")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
COURSE_MODE = "REFERENCE"  # <- edit once: REFERENCE | GPU_CLASSROOM | GPU_RESEARCH
RUN_OWN_GPU_ARMS = True   # <- False in GPU modes: reuse explicitly selected physical runs (ids shown + classified)
from lab import notebook_runtime as rt
from lab import notebook_support as ns
session = rt.begin(COURSE_MODE, notebook="06")

def _stamp(kind: str, value: str) -> None:
    """Print a visible origin tag so no number is mistaken for another kind."""
    print(f"[{kind}] {value}")

MEASURED_TODAY = "measured on THIS GPU in THIS session (physical)"
HISTORICAL   = "frozen reference evidence from the committed campaign"
ESTIMATE     = "analytical estimate or simulation - NOT a measurement"

from lab import student_provenance as prov

def _origin_stamp(category: str) -> str:
    """Stamp text for a provenance category; reuse is NEVER labelled as
    measured-today (fail-closed student provenance contract)."""
    return prov.stamp_text(category)

def gpu_cell(fn):
    """Run a physical exercise only in GPU modes; print why otherwise."""
    if session.mode != "REFERENCE":
        return fn()
    print("skipped physical exercise: COURSE_MODE=REFERENCE (no GPU work launched)")
    return None

def want_own_arms():
    """Whether this session executes its own bounded arms (True unless the
    student opted to reuse earlier physical runs to save GPU time)."""
    if session.mode == "REFERENCE" or RUN_OWN_GPU_ARMS:
        return True
    print("RUN_OWN_GPU_ARMS=False: reusing explicitly selected physical run(s) "
          "instead of launching new ones (metered-GPU mode; provenance shown)")
    return False


# Evidence selection (M5.1) — edit here, all in one place:
#   RUN_OWN_GPU_ARMS=True  : execute your own bounded arms on this GPU.
#   RUN_OWN_GPU_ARMS=False : paste EXACT run ids from an earlier session of
#                            this workspace to reuse them (they are classified
#                            SAME_STUDENT_PRIOR_PHYSICAL / SHARED_CAMPAIGN_
#                            PHYSICAL and never counted as new measurements);
#                            empty ids fall back to the committed reference.
REUSE_TE_RUNS = {"cuda": "", "cpu": "", "h3": ""}
REUSE_PIPELINE_RUN = ""

## 0 · 工件准入 —— PASS / PASS_WITH_WARNINGS / FAIL / BLOCKED
OV IR 会对照其转换清单做字节级校验（流式 SHA256、源模型摘要链、身份字段、保守 RAM 准入）。状态契约是严格的：

- `PASS` —— 所有身份/完整性/资源检查全部满足。
- `PASS_WITH_WARNINGS` —— 在已声明的范围内验证通过，但存在未锁定的历史完整性字段；绝不允许四舍五入成 PASS。
- `FAIL` —— 工件存在但未通过强制检查（摘要/大小/身份/清单）。依赖的推理必须停止。
- `BLOCKED` —— 工件或资源不可用；会打印补救命令。受影响的阶段标记为 BLOCKED，绝不静默跳过。

准入失败会**中止**本笔记本的物理阶段（fail closed）：绝不允许用未验证的工件产生性能数字。

In [ ]:
def ov_admission():
    from lab import m51_contract as contract
    res = contract.admit_ov_bf16()
    print("M5.1 OpenVINO BF16 artifact admission")
    print(f"  artifact_root: {res['artifact_root']}  ({res['resolved_from']})")
    for c in res["checks"]:
        print(f"  [{c['status']}] {c['name']}")
    for w in res.get("warnings", []):
        print(f"  warning: {w}")
    rg = res.get("ram_gate")
    if rg:
        print(f"  RAM: usable {rg.get('usable_bytes', 0)/2**30:.1f} GiB "
              f"(basis {rg.get('basis')}) vs need {rg['required_bytes']/2**30:.0f} GiB "
              f"-> {rg['status']}")
    q = res.get("cpu_quota") or {}
    print(f"  CPU quota: {q.get('cpus')} cores")
    print(f"  ADMISSION STATUS: {res['status']}")
    if res.get("remediation"):
        print(f"  remediation: {res['remediation']}")
    return res

OV_ADMIT = gpu_cell(ov_admission)
if OV_ADMIT is None:
    print("REFERENCE mode: admission executes on the measurement host. The "
          "qualification campaign recorded bf16 = PASS_WITH_WARNINGS (xml "
          "digest computed, not pinned by the historical manifest).")
elif OV_ADMIT["status"] not in ("PASS", "PASS_WITH_WARNINGS"):
    session.finish({"mode": session.mode, "notebook": session.notebook,
                    "session_id": session.session_id,
                    "physical_experiment_status": "BLOCKED",
                    "admission": {k: OV_ADMIT[k] for k in
                                  ("status", "artifact_root", "resolved_from",
                                   "remediation")}})
    raise rt.AdmissionError(
        f"M5.1 artifact admission {OV_ADMIT['status']}: the physical A/C "
        f"phases are BLOCKED. Remediation: {OV_ADMIT['remediation']}")

## A 部分 · 本地编码器：GPU PyTorch vs CPU PyTorch vs OpenVINO BF16
三个编码器、同一组提示词。当 `RUN_OWN_GPU_ARMS=True`（默认）时，本笔记本在**你的**主机上实测全部三个：GPU 探针、CPU PyTorch 探针，以及 H3 OpenVINO BF16 实验（完整锁定准入 + 各道门）。当 `RUN_OWN_GPU_ARMS=False` 时使用已提交的洁净室参考并如实标注。冻结参考比值（仅编码器、同一 CPU）：OV BF16 比 PyTorch CPU 快约 331–480×——请实测验证，不要想当然。

In [ ]:
enc = ns.read_json("runs/m51-20261105-cleanroom-01/performance/encoder_ab.json")
for arm in ("G_PyTorch_GPU", "C0_PyTorch_CPU", "C1b_OV_BF16"):
    row = enc["arms"][arm]["prompts"]["p1-zh-scene"]
    print(f"{arm:16s} warm median {row['warm_median_s']:8.3f}s  n={row['n_warm']}")
_pa = prov.classify_path("runs/m51-20261105-cleanroom-01/performance/encoder_ab.json")
_stamp(_origin_stamp(_pa["category"]),
       f"frozen encoder A/B (cleanroom campaign, 16-thread cgroup quota) "
       f"[{_pa['category']}, explicit path]")

def run_own_te_arms():
    te_gpu = session.run_m51_baseline("cuda", repeats=1)
    te_cpu = session.run_m51_baseline("cpu", repeats=1)
    h3_dir = session.run_h3("H3_OV_BF16")
    return te_gpu, te_cpu, h3_dir

TE = None

def _load_te_probe(run_id, dev):
    session.note_consumed(run_id, why=f"explicit reuse: TE probe {dev}")
    pr = json.loads((ROOT / "runs" / run_id / f"probe_{dev}.json").read_text())
    p = next(iter(pr["prompts"].values()))
    w = p["warm_samples_s"] if isinstance(p, dict) else [r["encode_s"] for r in p]
    return w[-1], pr.get("rss_peak_mb")

if OV_ADMIT is not None and want_own_arms():
    TE = gpu_cell(run_own_te_arms)
te = {"provenance": "COMMITTED_PHYSICAL_REFERENCE",
      "run_ids": ["m51-20261105-cleanroom-01"]}
if TE is not None:
    gpr = json.loads((TE[0] / "probe_cuda.json").read_text())
    cpr = json.loads((TE[1] / "probe_cpu.json").read_text())
    h3 = json.loads((TE[2] / "H3_RESULTS.json").read_text())
    gp = next(iter(gpr["prompts"].values()))
    cp = next(iter(cpr["prompts"].values()))
    g_w = gp["warm_samples_s"] if isinstance(gp, dict) else [r["encode_s"] for r in gp]
    c_w = cp["warm_samples_s"] if isinstance(cp, dict) else [r["encode_s"] for r in cp]
    ov = h3.get("measurements", {}).get("encode_s") or []
    ov = [s for s in ov[1:]] or ov  # drop the first (load-adjacent) sample if others exist
    te.update({"gpu_warm_s": g_w[-1], "cpu_warm_s": c_w[-1],
               "ov_bf16_s": statistics.median(ov) if ov else None,
               "cpu_rss_mb": cpr.get("rss_peak_mb"),
               "ov_rss_mb": h3.get("measurements", {}).get("rss_peak_mb"),
               "h3_status": h3.get("status"), "h3_gates": h3.get("gates", {}),
               "h3_mode": h3.get("mode"),
               "human_quality_reference": h3.get("human_quality_reference"),
               "provenance": "CURRENT_SESSION_PHYSICAL",
               "run_ids": [TE[0].name, TE[1].name, TE[2].name]})
    print(json.dumps({k: te.get(k) for k in ("gpu_warm_s", "cpu_warm_s",
                                             "ov_bf16_s", "cpu_rss_mb", "ov_rss_mb")},
                     indent=1))
    print("H3 gates:", json.dumps(te["h3_gates"], indent=1))
    if te["ov_bf16_s"]:
        _stamp(MEASURED_TODAY,
               f"encoder-only: PyTorch CPU {te['cpu_warm_s']:.1f}s -> OV BF16 "
               f"{te['ov_bf16_s']:.3f}s ({te['cpu_warm_s']/te['ov_bf16_s']:.0f}x) "
               f"on THIS host [run_id={TE[2].name}, mode={te['h3_mode']}]")
elif OV_ADMIT is not None and REUSE_TE_RUNS.get("h3"):
    h3 = json.loads((ROOT / "runs" / REUSE_TE_RUNS["h3"] /
                     "H3_RESULTS.json").read_text())
    ov = [s for s in h3.get("measurements", {}).get("encode_s", [])[1:]] or          h3.get("measurements", {}).get("encode_s", [])
    g_val, _ = (_load_te_probe(REUSE_TE_RUNS["cuda"], "cuda")
                if REUSE_TE_RUNS.get("cuda") else (None, None))
    c_val, c_rss = (_load_te_probe(REUSE_TE_RUNS["cpu"], "cpu")
                    if REUSE_TE_RUNS.get("cpu") else (None, None))
    enc2 = ns.read_json("runs/m51-20261105-cleanroom-01/performance/encoder_ab.json")
    te.update({"gpu_warm_s": g_val or enc2["arms"]["G_PyTorch_GPU"]["prompts"]["p1-zh-scene"]["warm_median_s"],
               "cpu_warm_s": c_val or enc2["arms"]["C0_PyTorch_CPU"]["prompts"]["p1-zh-scene"]["warm_median_s"],
               "ov_bf16_s": statistics.median(ov) if ov else None,
               "cpu_rss_mb": c_rss,
               "ov_rss_mb": h3.get("measurements", {}).get("rss_peak_mb"),
               "h3_status": h3.get("status"), "h3_gates": h3.get("gates", {}),
               "h3_mode": h3.get("mode"),
               "human_quality_reference": h3.get("human_quality_reference"),
               "provenance": "SAME_STUDENT_PRIOR_PHYSICAL",
               "run_ids": [r for r in REUSE_TE_RUNS.values() if r] or [REUSE_TE_RUNS["h3"]]})
    session.note_consumed(REUSE_TE_RUNS["h3"],
                          why="explicit reuse: OV BF16 encoder run")
    te["provenance"] = session.consumed[-1]["category"]
    _stamp(_origin_stamp(te["provenance"]),
           f"encoder numbers reused from your earlier physical runs "
           f"{te['run_ids']} (explicit ids; NOT new measurements here)")
else:
    row_g = enc["arms"]["G_PyTorch_GPU"]["prompts"]["p1-zh-scene"]["warm_median_s"]
    row_c = enc["arms"]["C0_PyTorch_CPU"]["prompts"]["p1-zh-scene"]["warm_median_s"]
    row_o = enc["arms"]["C1b_OV_BF16"]["prompts"]["p1-zh-scene"]["warm_median_s"]
    te.update({"gpu_warm_s": row_g, "cpu_warm_s": row_c, "ov_bf16_s": row_o})
    _stamp(_origin_stamp(te["provenance"]),
           "own TE arms not executed (RUN_OWN_GPU_ARMS=False or REFERENCE); "
           "encoder numbers stay on the committed reference above")

## B 部分 · 端到端物理 A/C —— 你自己的四请求批次
相同的 GPU、DiT、VAE、调度器、分辨率、提示词、种子与客户端策略；每个臂四个真实请求。当 `RUN_OWN_GPU_ARMS=True` 时，本笔记本在**你的** W7900 上**实际执行**两个臂，并按 run id 精确加载这些调用产生的记录（绝不使用最新文件通配）：

- **A：** GPU 原生 PyTorch TE + GPU DiT/VAE，顺序执行。
- **C：** OpenVINO BF16 CPU TE 与 GPU DiT/VAE 重叠执行。

诚实的指标是**批次墙钟时间 / 每分钟图像数**；服务延迟、CPU RSS 与 VRAM 单独报告。

In [ ]:
PIPE = None
if OV_ADMIT is not None and want_own_arms():
    def run_own_ac():
        return session.run_m51_pipeline(arms="A,C")
    PIPE = gpu_cell(run_own_ac)

pab = None
pab_origin = None
pab_run_id = None
if PIPE is not None:
    pab = json.loads((PIPE / "pipeline" / "pipeline_ab.json").read_text())
    pab_origin, pab_run_id = "CURRENT_SESSION_PHYSICAL", PIPE.name
elif OV_ADMIT is not None and REUSE_PIPELINE_RUN:
    session.note_consumed(REUSE_PIPELINE_RUN,
                          why="explicit reuse of an earlier session's A/C pipeline")
    pab = json.loads((ROOT / "runs" / REUSE_PIPELINE_RUN /
                      "pipeline" / "pipeline_ab.json").read_text())
    pab_origin = session.consumed[-1]["category"]
    pab_run_id = REUSE_PIPELINE_RUN
else:
    ppath = "runs/m51-20261105-cleanroom-01/pipeline/pipeline_ab.json"
    pab = ns.read_json(ppath)
    pc2 = prov.classify_path(ppath)
    pab_origin, pab_run_id = pc2["category"], "m51-20261105-cleanroom-01"

if pab is not None:
    A, C = pab["arms"]["A"], pab["arms"]["C"]
    print(f"A (GPU-native TE) : batch wall {A['total_s']:.2f}s | "
          f"{A['images_per_min']:.2f} img/min | warmup {A['server_warmup_s']:.1f}s | "
          f"vram_peak {A['vram_peak_mb']:.0f}MB | server_rss {A['server_rss_after_mb']:.0f}MB")
    print(f"C (OV BF16 CPU TE): batch wall {C['total_s']:.2f}s | "
          f"{C['images_per_min']:.2f} img/min | worker_load {C['worker_load_s']:.1f}s | "
          f"fill {C['fill_s']:.1f}s | worker_rss {C['worker_rss_after_mb']:.0f}MB | "
          f"server_rss {C['server_rss_after_mb']:.0f}MB")
    print(f"C overlap proven: {C['overlap_proven']} "
          f"(total overlap {C['overlap_s_total']:.2f}s of CPU encode under GPU work)")
    _stamp(_origin_stamp(pab_origin),
           f"A/C pipeline evidence [run_id={pab_run_id}]")
else:
    pab = ns.read_json("runs/m51-20261105-cleanroom-01/pipeline/pipeline_ab.json")
    pc2 = prov.classify_path("runs/m51-20261105-cleanroom-01/pipeline/pipeline_ab.json")
    pab_origin, pab_run_id = pc2["category"], "m51-20261105-cleanroom-01"
    _stamp(_origin_stamp(pab_origin),
           "committed cleanroom pipeline A/B (explicit path; no own arms "
           "executed here)")

## E 部分 · 用证据解读 A/C 结果
先看工作负载身份（臂匹配是前提，不是假设），然后是五个证据视图——每个都引用精确的 run id。仅编码器加速与系统加速是两个**不同**的数字。

In [ ]:
wl = prov.workload_identity(pab) if pab else None
if pab is not None:
    A, C = pab["arms"]["A"], pab["arms"]["C"]
    print(f"workload identity [{pab_run_id}]: {wl['n_requests']} requests, "
          f"seed {wl['seed']}, prompt_digest {wl['prompt_digest']}")
    a_srv = statistics.median([r["wall_s"] for r in A["requests"]])
    c_gen = statistics.median([r["gen_wall_s"] for r in C["requests"]])
    c_enc = statistics.mean([r.get("cpu_encode_s", 0.0) for r in C["requests"]])
    c_wait = sum(r.get("wait_after_gen_s", 0.0) for r in C["requests"])
    print(f"service latency (median/request): A {a_srv:.2f}s | C {c_gen:.2f}s")
    print(f"C CPU encode mean {c_enc:.3f}s/request; post-gen handoff wait "
          f"{c_wait:.2f}s total")
    e2e = A["total_s"] / C["total_s"]
    print(f"system batch speedup (C vs A): {e2e:.3f}x  "
          f"[{A['total_s']:.2f}s -> {C['total_s']:.2f}s]")
    _stamp(_origin_stamp(pab_origin),
           f"system batch speedup {e2e:.3f}x from run_id={pab_run_id}")
    if te.get("ov_bf16_s") and te.get("cpu_warm_s"):
        enc_only = te["cpu_warm_s"] / te["ov_bf16_s"]
        where = ("this host" if te["provenance"] == "CURRENT_SESSION_PHYSICAL"
                 else "committed reference")
        print(f"encoder-only speedup ({where}): {enc_only:.0f}x  "
              "<- NOT the system speedup; explain the difference")
    if PIPE is not None:
        try:
            import matplotlib.pyplot as plt
            fig, axes = plt.subplots(1, 5, figsize=(16, 2.7))
            data = [
                ("batch wall (s)", [A["total_s"], C["total_s"]], "AB"),
                ("images / min", [A["images_per_min"], C["images_per_min"]], None),
                ("CPU-side RSS (MB)", [C["worker_rss_after_mb"],
                                       C["server_rss_after_mb"]], "CC"),
                ("GPU VRAM peak (MB)", [A["vram_peak_mb"], 0], "A0"),
                ("speedup: encoder vs system",
                  [te["cpu_warm_s"] / te["ov_bf16_s"], e2e], "log"),
            ]
            for ax, (title, vals, note) in zip(axes, data):
                ax.bar(["A", "C"] if len(vals) == 2 and note != "CC"
                       else ["C-worker", "C-server"] if note == "CC"
                       else ["A", "C"], vals, color=["#4477aa", "#ee6677"])
                if note == "log":
                    ax.set_yscale("log")
                ax.set_title(f"{title}", fontsize=8)
                for i, v in enumerate(vals):
                    ax.text(i, v, f"{v:.3g}", ha="center", va="bottom", fontsize=7)
            fig.suptitle(f"M5.1 physical A/C evidence | run_id={PIPE.name} | "
                         f"CURRENT_SESSION_PHYSICAL", fontsize=9)
            fig.tight_layout()
            plt.show()
        except Exception as e:
            print("charts skipped:", e)
        print("NOTE: arm C's peak GPU VRAM is not recorded by the pipeline "
              "harness; the C bar above is 0 by absence of evidence, not a "
              "measured zero. A-side peak is measured.")

## C 部分 · 公平因子实验 —— 架构 × 客户端监控
拿 C-fast 对比 A-original 会夸大架构效应：A-original 还要支付每请求的监控器拆除成本。2×2 因子设计（A/C × original/fast）把两个轴分开。课堂模式阅读已提交的 clean-HEAD 实验（显式来源、带来源标注）；设置 `RUN_OWN_FAIR_AB = True` 可执行你自己的有界因子实验（pairs=1、blocks=1，约 20+ 分钟）。旧研究发现：客户端匹配时架构收益*较小*，而客户端效应更大——把它们当作假设。

In [ ]:
RUN_OWN_FAIR_AB = False  # True (advanced, ~20+ min): own bounded 2x2 factorial

ff_source = None
cells = None
if RUN_OWN_FAIR_AB and OV_ADMIT is not None and session.mode != "REFERENCE":
    fa_dir = session.run_fairab(pairs=1, blocks=1)
    prof_own = json.loads((fa_dir / "pipeline_profile" /
                           "pipeline_profile.json").read_text())
    cells = prof_own["analysis"]["views"]["primary_all_batches"]
    ff_source = ("CURRENT_SESSION_PHYSICAL", fa_dir.name)
else:
    ff = ns.fair_factorial(clean_head=False)
    cells = ff["cells"]
    ff_source = ("COMMITTED_PHYSICAL_REFERENCE", ff["source"])
print(f"fair factorial source: {ff_source[1]} [{ff_source[0]}]")
print(f"{'cell':12s} {'n':>2s} {'median_s':>9s} {'img/min':>8s}")
for k, v in cells.items():
    if not isinstance(v, dict) or "median_s" not in v:
        continue
    print(f"{k:12s} {v['n']:2d} {v['median_s']:9.2f} {v.get('images_per_min', v.get('img_per_min')):8.2f}")
arch = cells["A_original"]["median_s"] / cells["C_original"]["median_s"]
arch_fair = cells["A_fast"]["median_s"] / cells["C_fast"]["median_s"]
client_a = cells["A_original"]["median_s"] / cells["A_fast"]["median_s"]
client_c = cells["C_original"]["median_s"] / cells["C_fast"]["median_s"]
_stamp(_origin_stamp(ff_source[0]),
       f"fair factorial ({ff_source[1]}): architecture effect {arch:.3f}x "
       f"(original clients) vs {arch_fair:.3f}x (matched fast clients); "
       f"client effect A {client_a:.3f}x / C {client_c:.3f}x")
print("never report C_fast vs A_original as a pure architecture effect")

## 正确性范围 —— 什么保持 FAIL，什么留给人工
1. INT8 编码器臂的数值 **FAIL 永远是 FAIL**——本课程任何地方都不会自动提升质量结论。
2. C1b 所有者验收只限定于**历史评估过的部署**；你的新物理运行需要自己的正确性证据。

> **质量判定永远由人完成。** 任何单元都不会自动批准图像质量结论。历史所有者判定**只**适用于其原始评估的部署范围；新的物理运行需要自己的评审（在此之前保持 `PENDING_HUMAN_REVIEW`）。

## 动手练习
1. 用上面的单元计算“仅编码器加速比”与“流水线加速比”——为什么差一个数量级？
2. Part C 中你会给哪个格子加样以收紧架构效应估计？为什么？
3. 在 55 GiB cgroup 上跑 PyTorch CPU 完整臂之前，你会做怎样的 RAM 准入检查？

## G 部分 · 你的结论 —— 用你自己的证据回答
用上面的 run id 和数字作答（要引用它们）：
1. **编码器本身**加速了多少（PyTorch CPU → OV BF16）？
2. **端到端系统**加速了多少（A → C 批次墙钟）？
3. 为什么两个收益相差若干个数量级？
4. C 架构释放了多少 GPU 显存？你可以用它做什么？
5. C worker 消耗了多少 CPU 内存？是哪道准入检查在保护它？
6. 在**这套测量协议**下，优化后的架构确实更好吗？范围如何（n、样本总体）？
7. 还有哪些不确定性？下一个你会补做什么实验？

In [ ]:
_OWN = PIPE is not None
_PRIOR = (not _OWN and pab_origin in ("SAME_STUDENT_PRIOR_PHYSICAL",
                                      "SHARED_CAMPAIGN_PHYSICAL"))
_PHYS = _OWN or _PRIOR
m51_result = {
    "experiment_id": "M51",
    "session_id": session.session_id,
    "course_mode": session.mode,
    "run_ids": ([PIPE.name] if _OWN else
                [pab_run_id] if _PRIOR else list(te.get("run_ids", []))),
    "provenance": ("CURRENT_SESSION_PHYSICAL" if _OWN else
                   pab_origin if _PRIOR else
                   te.get("provenance", "COMMITTED_PHYSICAL_REFERENCE")),
    "execution_status": ("MEASURED" if _OWN else "REUSED"),
    "gpu": {"arch": (session.admission.detail.get("gfx_arch")
                    if session.admission and session.admission.detail else None),
            "model": (session.admission.detail.get("device_name")
                      if session.admission and session.admission.detail else None)},
    "source_head": session.source.get("head"),
    "source_dirty": session.source.get("dirty"),
    "workload": prov.workload_identity(pab) if pab else {"note": "no record loaded"},
    "populations": {"warmup_count": 1, "measured_count": 4,
                    "cold_warm": "warm-resident batch"},
    "measurements": ({ "a_batch_wall_s": pab["arms"]["A"]["total_s"],
                       "c_batch_wall_s": pab["arms"]["C"]["total_s"],
                       "a_img_per_min": pab["arms"]["A"]["images_per_min"],
                       "c_img_per_min": pab["arms"]["C"]["images_per_min"],
                       "encoder_gpu_warm_s": te.get("gpu_warm_s"),
                       "encoder_cpu_warm_s": te.get("cpu_warm_s"),
                       "encoder_ov_bf16_s": te.get("ov_bf16_s")}
                      if pab else {"note": "reference-only session"}),
    "throughput": ({"numerator": 4, "denominator": pab["arms"]["C"]["total_s"],
                    "unit": "images/s (C arm)"}
                   if pab and _PHYS else None),
    "speedup": (round(pab["arms"]["A"]["total_s"] / pab["arms"]["C"]["total_s"], 3)
                if pab and _PHYS else None),
    "memory": ({"cpu_worker_rss_mb": pab["arms"]["C"].get("worker_rss_after_mb"),
                "cpu_server_rss_mb": pab["arms"]["C"].get("server_rss_after_mb"),
                "gpu_vram_peak_mb": pab["arms"]["A"].get("vram_peak_mb"),
                "encoder_cpu_rss_mb": te.get("cpu_rss_mb")}
               if pab else {"note": "reference-only session"}),
    "numerical_status": (te.get("h3_gates", {}).get("conditioning_alignment")
                         or "NOT_ASSESSED"),
    "human_quality_status": "PENDING_HUMAN_REVIEW",
    "classification": ("BOUNDED_PHYSICAL_COMPARISON" if _PHYS
                       else "REFERENCE_ONLY"),
    "ac_pair": ({"a_run_id": pab_run_id, "c_run_id": pab_run_id,
                 "a_workload": prov.workload_identity(pab),
                 "c_workload": prov.workload_identity(pab)}
                if _PHYS else None),
    "result_status": "PASS",
    "source_paths": ([f"runs/{pab_run_id}/pipeline/pipeline_ab.json"]
                     if _PHYS else
                     ["runs/m51-20261105-cleanroom-01/pipeline/pipeline_ab.json"]),
    "reproduce_command": ("python -u lab/m51_pipeline.py --run <fresh-tag> "
                          "--arms A,C"),
}
prov.validate_or_raise(m51_result, session_id=session.session_id)
print("[STUDENT RESULT VALIDATED]",
      m51_result["experiment_id"], m51_result["provenance"],
      m51_result["classification"])

summary = {
    "mode": session.mode,
    "notebook": session.notebook,
    "session_id": session.session_id,
    "executions": [e.get("run_id") or e.get("kind") for e in session.executions],
    "consumed": [c.get("run_id") for c in session.consumed],
    "physical_experiment_status": ("MEASURED" if _OWN else
                                   "REUSED_PRIOR_PHYSICAL" if _PRIOR else
                                   "REFERENCE_ONLY"),
    "student_result": m51_result,
    "note": "append-only session record; physical evidence lives under runs/",
}
session.finish(summary)

## 验收标准

- [ ] 你能指出哪些数字是**今天在这块 GPU 上实测的**、哪些是
      **历史参考值**、哪些是**估计/模拟**。
- [ ] 你产生的每次物理运行都在 `runs/` 下有可见的 `run_id`。
- [ ] 你没有把 smoke/课堂样本冒充为正式协议结果。
- [ ] 你能解释你报告的每项测量的一个局限性。

## 复现与贡献

```bash
python -m lab doctor                                  # 准入证据 -> runs/doctor.json
python -m lab run --experiment H3_OV_BF16 --date <your-tag>  # 完整研究协议
python -m lab report --scope student --kind all       # 只汇总你自己的 runs
```

提交你的 `runs/<run-id>/*.json` 记录（绝不写 `evidence/`），并在 PR 中说明：
假设 → 协议 → 数字 → 局限。欢迎负结果——测得的退化是科学结果，不是失败。